# An introduction to test-driven data analysis (TDDA), using EM-DAT disaster data
This tutorial explores how one can apply TDDA principles to the EM-DAT disaster data.
It uses only packages such as pandas, matplotlib and seaborn. It does not use the TDDA library.
The outline of the tutorial is the following:

| # | Part | Min |
|---|---|---|
| 1 | Why validate data? | 3 |
| 2 | What is TDDA? | 3 |
| 3 | Hands-on: EM-DAT, following the book's checklists | 22 |
| | &nbsp;&nbsp;• a brief look at the data | 2 |
| | &nbsp;&nbsp;• **NDS**: the *new data source* checklist, items 1–6 | 9 |
| | &nbsp;&nbsp;• **DV**: the *data validation* checklist, items 1–6 | 11 |
| 4 | Wrap-up and where next | 2 |

<summary>Before we start:</summary>

* EM-DAT is free for non-commercial use after registration. Check its terms before sharing the raw file, and credit it: *EM-DAT, CRED / UCLouvain, Brussels, Belgium, www.emdat.be*.
* * Log in and download data from EM-DAT website. This tutorial relies on data from 2000 to 2025 from all countries and natural disasters.
* Method and checklists: Nicholas J. Radcliffe, [*Test-Driven Data Analysis*](https://book.tdda.info/book/index.html). Section numbers below (📖) refer to that book. The checklists are openly licensed (CC BY 4.0): [book.tdda.info/checklists](https://book.tdda.info/checklists/).



---
# 1 · Why validate data?

Every story built on data inherits that data's mistakes. The dangerous mistakes are the **quiet** ones: the code runs, the chart looks fine, but the number is wrong. 📖 *§2.1 GIGO (garbage in, garbage out)*

Quiet mistakes tend to come in four flavours:

| Kind of problem | What it looks like |
|---|---|
| **Read wrongly** | numbers stored as text, text silently turned into blanks, a column dropped or renamed |
| **Misunderstood** | one row is not one event; a blank is not a zero |
| **Impossible or suspicious values** | typos, swapped columns |
| **Drift** | next month's download has new categories, a new coding scheme |

Two facts make this worth automating:

1. You will **download the same dataset again** (EM-DAT is updated regularly). Checking by eye once is fine; checking by eye every time is not.
2. The checks that matter are usually *simple statements about the data* ("this column is never blank", "this code is one of these 39 values"). A computer can run those in milliseconds.

The attitude the book asks of every analyst is a single question to keep asking: **"How is this misleading data misleading me?"**

**Today's goal:** take a spreadsheet we have never seen, and in 30 minutes turn it into *checks that run again on every new release*.

---
# 2 · What is TDDA?

**Test-driven development (TDD)** in software means writing tests that say what the code *should* do, and re-running them all the time. **Test-driven data analysis (TDDA)** brings that habit to data work: write down what you *expect* of your data and your results, as tests a computer can run. 📖 *§1.2 From TDD to TDDA*

The book has two main toolkits:

| Toolkit | Question it answers | Today? |
|---|---|---|
| **Data validation with constraints** (Part I) | "Is this *input* data what I think it is?" | ✅ yes |
| **Reference tests** (Part II) | "Did my *analysis output* change when it shouldn't have?" | no |

### The idea of a constraint
A **constraint** is a statement that should be true of the data, for example:

* `Country` is one of the N values we have seen before
* `Total Deaths`, when filled in, is at least 1
* `DisNo.` always looks like `9999-9999-AAA`

The companion Python package (`tdda`) has three verbs, and we'll hand-roll the same three:

| Verb | What it does |
|---|---|
| **discover** | scan example "good" data and *propose* constraints that hold on it |
| **verify** | check a dataset against constraints and count the failures |
| **detect** | list the individual records that fail |

Two cautions that shape the whole tutorial:

* **Discovered constraints describe the data, not the truth.**
* **A failing constraint means "look here"**, not "this is wrong".
  
### Our route: two checklists from the book
| Checklist | Items we follow | 📖 |
|---|---|---|
| **NDS**: handling a *new data source* | 1 learn from the owner · 2 explore · 3 examine metadata · 4 data dictionary and conversions · 5 profile and audit · 6 note what looks odd, then investigate | §7.1, App. C.5 |
| **DV**: *data validation* | 1 read the data correctly · 2 data dictionary · 3 profile and audit · 4 create constraints on known-good data · 5 validate and refine them · 6 check new data | App. C.4 |

We stop at DV6; DV7–DV13 (custom constraints, roll-ups, drift, …) are the natural next steps.

<summary>Note on tools</summary>

The `tdda` package implements all of this (`tdda discover`, `tdda verify`, `tdda detect`) with a wider range of constraint types.